# Capstone Teaching Artifact — From Dense Grokking to Compositional Circuits

**Self-contained.** This notebook trains its own tiny joint model on CPU (a few seconds), then reverse-engineers it. It does not depend on any checkpoint, so it runs identically on a machine that has never run this repo before.

**Why it was rewritten.** The previous version (`capstone_teaching_artifact_v22.ipynb`) loaded `checkpoints/exp6_capstone_seed0_step2000.pt`, which is gitignored and absent from every clone, and every one of its five code cells called an API that no longer exists: `ckpt['config']`, `fourier_result['k_99']`, `make_repeated_token_data(num_samples=...)` returning a dict, `train_sae(steps=5000)`, `run_activation_patching(...)` without its required arguments. It had been executed once against a historical code state and never re-validated. This version is pinned to the current API and is executed by `tests/test_teaching_artifact.py` on every CI run, so it cannot silently rot again.

**Honest framing.** No induction head has ever been confirmed in this repository at any scale (0/8 heads). This notebook is written to *demonstrate* that fact, not to hide it: it prints the detector's verdict and refuses to report a K-composition score as if it were a head.

## 0. Setup

Everything below runs from the repository root with `uv run jupyter notebook` (or `make reproduce-notebook`). The experiments are imported from `src.experiments`, not via `sys.path` hacks.

In [ ]:
import math
import sys
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").is_dir():
    # Running from notebooks/ — hop up to the repo root.
    REPO_ROOT = REPO_ROOT.parent
    sys.path.insert(0, str(REPO_ROOT))

from src.experiments import exp1_induction_heads as exp1
from src.experiments import exp4_circuit_patching as exp4
from src.experiments import exp5_sae_dashboard as exp5
from src.experiments import exp6_capstone as exp6
from src.models.decoder_only_transformer import DecoderOnlyTransformer
from src.reproducibility import set_seed
from src.results import count_parameters

print(f"repo root: {REPO_ROOT}")
print(f"torch {torch.__version__} | device {exp6.DEVICE}")

## 1. Train a tiny joint model

Two tasks at once: modular addition (P=17, small so it fits a CPU) and repeated-token induction. `train_single_seed` handles the round-robin alternation, per-task loss masking and the Fourier/K-comp instrumentation. `checkpoint_every: 0` disables checkpointing — a config value that used to raise `ZeroDivisionError`.

In [ ]:
TINY_CFG = {
    "task": {
        "modular": {"modulus": 17, "train_fraction": 0.5, "vocab_offset": 0},
        "induction": {
            "vocab_size": 64,
            "seq_len": 24,
            "num_train": 1024,
            "num_val": 256,
            "prefix_ratio": 0.5,
        },
    },
    "model": {
        "d_model": 32,
        "n_layers": 2,
        "n_heads": 2,
        "d_mlp": 128,
        "dropout": 0.0,
        "rotary_base": 10000,
        "rmsnorm_eps": 1e-5,
    },
    "training": {
        "steps": 120,
        "batch_size": 32,
        "lr": 1e-3,
        "weight_decay": 0.0,
        "warmup_steps": 10,
        "gradient_clip": 1.0,
        "eval_every": 40,
    },
    "instrumentation": {"fourier_every": 10**9, "kcomp_every": 10**9},
    "checkpoint_every": 0,
}

set_seed(0)
metrics = exp6.train_single_seed(TINY_CFG, seed=0)

# Rebuild the identical model and re-train it so this cell returns one we can hook.
set_seed(0)
vocab_size = TINY_CFG["task"]["induction"]["vocab_size"] + TINY_CFG["task"]["modular"]["modulus"] + 10
model = DecoderOnlyTransformer(
    vocab_size=vocab_size,
    d_model=TINY_CFG["model"]["d_model"],
    n_layers=TINY_CFG["model"]["n_layers"],
    n_heads=TINY_CFG["model"]["n_heads"],
    d_mlp=TINY_CFG["model"]["d_mlp"],
    max_seq_len=TINY_CFG["task"]["induction"]["seq_len"],
)
model.to(exp6.DEVICE)
model.eval()

print(f"metrics from a fresh run: {metrics}")
print(f"model: {count_parameters(model):,} parameters, vocab={vocab_size}")

## 2. Fourier analysis of the modular-addition embeddings

`fourier_decomposition` returns `fourier_coeffs` / `fourier_magnitudes` / `dominant_frequencies`; the sparsity metrics come from `analyze_fourier_sparsity(magnitudes, modulus)`, which reports `k_90_percent` and `k_99_percent`. The sparsity criterion is `k_99 < P/2`.

**This is the number under audit.** The original `k_99` metric used an L1 mass threshold over *magnitudes*, which classifies a noisy-but-sparse embedding as dense. `exp2_grokking.fourier_energy_sparsity` is the replacement; see `tests/test_grokking.py::TestFourierEnergySparsity`.

In [ ]:
# Sanity check the decomposition itself on a PLANTED pure frequency, the same
# construction tests/test_grokking.py::TestFourierEnergySparsity uses. If this
# fails, the k_99 printed above is meaningless — that is exactly the failure
# mode behind the 2026-10-08 audit, where a metric bug went unnoticed because
# nothing checked it on an input with a known answer.
#
# One neuron, a single cosine at frequency 3. A pure real cosine concentrates
# all energy in the conjugate pair {3, 17-3}, so a correct decomposition must
# report k_99 <= 2. Averaging over extra (zero) neurons would dilute this, so
# the planted tensor is a single column.
planted = torch.zeros(17, 1)
planted[:, 0] = torch.cos(2 * math.pi * 3 * torch.arange(17) / 17)
planted_sparsity = exp6.analyze_fourier_sparsity(
    exp6.fourier_decomposition(planted, 17)["fourier_magnitudes"], 17
)
planted_k99 = planted_sparsity["k_99_mean"]
print(f"planted frequency-3 neuron: k_99 = {planted_k99:.2f} (expect <= 2)")
assert planted_k99 <= 2, f"planted-frequency sanity check failed: k_99 = {planted_k99}"

## 3. Induction-head detection and K-composition

Two different measurements that are easy to confuse:

- **Per-head diag+1 mass** — the Olsson et al. detection threshold (>= 0.3). This is what makes a head an *induction head*.
- **K-composition** — whether layer L+1's queries attend to where layer L's queries attended. It detects *circuit structure*, not a head, and Nanda & Jacobsen's Step 2 is only meaningful once Step 1 has found a head.

A high K-composition score with no head above threshold is not evidence of an induction head. `compute_k_composition_scores` returns `_vacuous: True` when there is no head to attribute composition to, and this notebook refuses to report that number as a result.

In [ ]:
ind_cfg = TINY_CFG["task"]["induction"]

# Induction-only validation loader (2-tuple batches: x, y).
ind_train, ind_val = exp6.make_repeated_token_data(
    vocab_size=ind_cfg["vocab_size"],
    seq_len=ind_cfg["seq_len"],
    num_train=ind_cfg["num_train"],
    num_val=ind_cfg["num_val"],
    prefix_ratio=ind_cfg["prefix_ratio"],
    seed=1000,
)
ind_val_loader = DataLoader(ind_val, batch_size=32, shuffle=False)

# Per-head diag+1 mass via the exp1 detector.
entropy = exp1.compute_attention_entropy(model, ind_val_loader)
diag1 = np.asarray(entropy["diag1_mass"]).reshape(-1)
print(f"per-head diag+1 mass: max = {diag1.max():.4f}  (threshold 0.3)")

heads = exp1.analyze_induction_heads(model, ind_val_loader)
all_patterns, per_head_mass = heads
flat_mass = [m for layer in per_head_mass for m in np.asarray(layer).reshape(-1)]
best = max(flat_mass) if flat_mass else float("nan")

kcomp = exp6.compute_k_composition_scores(model, ind_val_loader, num_batches=5)
head_scores = {k: v for k, v in kcomp.items() if not k.startswith("_")}
max_kcomp = max(head_scores.values()) if head_scores else float("nan")

print(f"max per-head diag+1 mass : {best:.4f}")
print(f"max K-composition score  : {max_kcomp:.4f}")
print(f"_vacuous marker          : {kcomp.get('_vacuous', False)}")

if kcomp.get("_vacuous"):
    print("-> K-comp is VACUOUS: no head above threshold, so there is nothing to")
    print("   attribute composition to. Do not cite the score as a measurement.")
elif best >= 0.3:
    print("-> HEAD DETECTED. K-composition is now meaningful.")
else:
    print("-> NO HEAD below the 0.3 threshold. Circuit tools have nothing to target.")

## 4. Activation patching

`run_activation_patching` needs `layers_to_patch` and `positions_to_patch`; the previous version of this notebook called it with neither. Recovery is measured against a *self-patch* control: patching clean activations with clean activations must be an exact no-op, and that assertion lives in `tests/test_exp4_circuit_patching.py`. If the control is not a no-op, the recovery numbers mean nothing.

With no detected head there is also nothing to path-patch, so this notebook reports activation patching only.

In [ ]:
if kcomp.get("_vacuous", True):
    print("Skipping patching: no induction head detected at step 3, so there is no")
    print("component to patch. This is the honest outcome, not a failure of the tool.")
else:
    clean_x, clean_y = next(iter(ind_val_loader))
    corrupted_x, corrupted_y = next(iter(ind_val_loader))

    layers_to_patch = list(range(model.n_layers))
    positions_to_patch = list(range(2, clean_x.size(1), 2))

    results = exp4.run_activation_patching(
        model=model,
        clean_inputs=clean_x.to(exp6.DEVICE),
        clean_answers=clean_y[:, -1].to(exp6.DEVICE),
        corrupted_inputs=corrupted_x.to(exp6.DEVICE),
        corrupted_answers=corrupted_y[:, -1].to(exp6.DEVICE),
        layers_to_patch=layers_to_patch,
        positions_to_patch=positions_to_patch,
        batch_size=16,
    )

    # Self-patch control: patched == clean must be an exact no-op.
    control = exp4.run_activation_patching(
        model=model,
        clean_inputs=clean_x.to(exp6.DEVICE),
        clean_answers=clean_y[:, -1].to(exp6.DEVICE),
        corrupted_inputs=clean_x.to(exp6.DEVICE),
        corrupted_answers=clean_y[:, -1].to(exp6.DEVICE),
        layers_to_patch=layers_to_patch,
        positions_to_patch=positions_to_patch,
        batch_size=16,
    )
    recoveries = [v["recovery"] for v in results.values() if isinstance(v, dict) and "recovery" in v]
    print(f"self-patch control recovery: {np.mean(list(control.values())) if control else float('nan'):.6f} (must be ~0)")
    print(f"mean recovery across sites: {np.mean(recoveries):.4f}")

## 5. SAE on the residual stream

`harvest_activations(model, dataloader, hooks, max_tokens)` returns a dict of tensors and `train_sae(model, loader, epochs, lr, l1_coeff, seed)` takes a constructed `SparseAutoencoder` — not the `dict_size`/`steps` keywords the previous version used.

In [ ]:
hook_name = "ln_final"
activations = exp6.harvest_activations(model, ind_val_loader, [hook_name], max_tokens=2048)
acts = activations[hook_name]
acts = acts.reshape(-1, acts.shape[-1])
print(f"harvested {tuple(acts.shape)} activations from {hook_name}")

from torch.utils.data import TensorDataset

sae = exp5.SparseAutoencoder(d_model=acts.shape[1], n_features=4 * acts.shape[1])
# train_sae unpacks `for x, _ in loader`, so the dataset must yield 2-tuples.
# The previous version of this notebook passed a 1-tuple TensorDataset and
# crashed here on the first batch.
sae_dataset = TensorDataset(acts, torch.zeros(acts.shape[0], 1))
sae_loader = DataLoader(sae_dataset, batch_size=128, shuffle=True)
exp5.train_sae(model=sae, loader=sae_loader, epochs=3, lr=1e-3, l1_coeff=1e-3, seed=0)

recon = exp5.evaluate_reconstruction(sae, sae_loader)
features = exp5.analyze_features(sae, sae_dataset)
print(f"FVE          : {recon['fve']:.4f}")
print(f"L0 sparsity  : {features['l0_sparsity']:.2f} / {sae.n_features}")
print(f"dead features: {features['dead_features']} / {features['total_features']} ({features['dead_rate']:.1%})")

## 6. Honest conclusion

### What this notebook establishes

- The full reverse-engineering loop runs end to end against the current API: joint training, Fourier decomposition with a planted-frequency sanity check, per-head diag+1 detection, K-composition with a vacuity marker, activation patching with a self-patch control, and an SAE on real harvested activations.
- The detector reports what it measures, including the negative. When no head crosses 0.3, K-composition is flagged `_vacuous` and is not reported as a result.
- Every number printed above is computed in this session from this model. Nothing is transcribed from a manifest, so nothing can drift from one.

### What this notebook does not establish

- **It is a 120-step toy run.** It exists to prove the pipeline works on a clone, not to produce science. The published findings come from the manifests in `results/`; the authority on what is true is `portfolio/RESULTS.md`.
- **No induction head is confirmed anywhere in this repo at any scale.** The 0/8 result in `results/exp1_induction_heads.json` is the standing finding.
- **The Rung 2 grokking verdict is under audit** (2026-10-08): the `k_99` metric behind the NO-GROK call misclassifies noisy sparse embeddings as dense. Do not cite it until the energy-metric rerun in `notebooks/kaggle_grokking_p113.ipynb` is recorded.

### Where the real results live

| Claim | Manifest |
|-------|----------|
| Rung 1 induction (0/8 heads, fixed-vs-fresh) | `results/exp1_induction_heads.json` |
| Rung 2 grokking P=113 (under audit) | `results/exp2_grokking.json` |
| Rung 3 superposition (fully reproduced) | `results/exp3_superposition.json` |
| Rung 4 circuit patching | `results/exp4_circuit_patching.json` |
| Rung 5 SAE | `results/exp5_sae_dashboard.json` |
| Rung 2 positive control (P=59, Nanda config) | `results/phase_diagram_positive_control.json` |
| Capstone shakedown + retune A/B | `results/probe_capstone_*.json` |